In [59]:
from pyomo.environ import *

In [60]:
solver = SolverFactory('gurobi')

In [61]:
r = ['solar', 'wind', 'power', 'water', 'hydrogen', 'oxygen']
o = ['PV', 'WF', 'PEM']


c_solar = 1.2  # - for power
c_wind = 1.3  # - for power
c_power = 3.9  # -
c_water = 2.95  # -
c_hydrogen = 1  # +
c_oxygen = 4.7  # +
# for hydrogen

m_solar = 0.1  # -
m_wind = 0.2  # -
m_power_solar = 1.1  # -
m_power_wind = 1.2  # -
m_water = 0.4  # -
m_hydrogen = 3  # +
m_oxygen = 1.2  # +

In [62]:
m = ConcreteModel()
m.m = Var(['c', 'p', 'd'], within=NonNegativeReals)
m.c = Var(['solar', 'wind', 'water'], within=NonNegativeReals)
m.d = Var(['hydrogen', 'oxygen'], within=NonNegativeReals)
m.p = Var(['power-solar', 'power-wind', 'hydrogen', 'oxygen'], within=NonNegativeReals)

m.con1 = Constraint(expr=m.p['hydrogen'] - m.d['hydrogen'] == 0)
m.con2 = Constraint(expr=m.p['oxygen'] - m.d['oxygen'] == 0)
m.con3 = Constraint(expr=m.c['water'] - c_water * m.p['hydrogen'] == 0)
m.con4 = Constraint(expr=m.d['oxygen'] - c_oxygen * m.p['hydrogen'] == 0)
m.con5 = Constraint(
    expr=(m.p['power-solar'] + m.p['power-wind']) / c_power - m.p['hydrogen'] == 0
)
m.con61 = Constraint(expr=(1 / c_solar) * m.c['solar'] - m.p['power-solar'] == 0)
m.con62 = Constraint(expr=(1 / c_wind) * m.c['wind'] - m.p['power-wind'] == 0)
m.con7 = Constraint(expr=m.c['water'] <= 2.6)
m.con8 = Constraint(expr=m.c['solar'] <= 0.9)
m.con9 = Constraint(expr=m.c['wind'] <= 0.65)
m.con10 = Constraint(expr=m.d['oxygen'] <= 1.5)
m.con11 = Constraint(
    expr=m_solar * m.c['solar'] + m_wind * m.c['wind'] + m_water * m.c['water']
    == m.m['c']
)
m.con12 = Constraint(
    expr=m_power_solar * m.p['power-solar'] + m_power_wind * m.p['power-wind']
    == m.m['p']
)
m.con13 = Constraint(
    expr=m_hydrogen * m.d['hydrogen'] + m_oxygen * m.d['oxygen'] == m.m['d']
)

m.obj = Objective(expr=m.m['c'] + m.m['p'] - m.m['d'])


res_m = solver.solve(m)

In [1]:
len(2)

TypeError: object of type 'int' has no len()

In [63]:
m.obj()

-0.7436170212765958

In [64]:
print('cost:', m.obj())
print('cost-c', m.m['c']())
print('cost-p', m.m['p']())
print('cost-d', m.m['d']())

cost: -0.7436170212765958
cost-c 0.5952127659574469
cost-p 1.4186170212765956
cost-d 2.7574468085106383


In [68]:
w = ConcreteModel()

w.x_o = Var(['solar', 'wind', 'water'], within=NonNegativeReals)
w.x_i = Var(['oxygen', 'hydrogen', 'power-solar', 'power-wind'], within=NonNegativeReals)

w.con2 = Constraint(expr=w.x_o['solar'] <= m_solar * 0.9 / (c_power * c_solar))
w.con3 = Constraint(expr=w.x_o['wind'] <= m_wind * 0.65 / (c_power * c_wind))
w.con4 = Constraint(expr=w.x_o['water'] <= m_water * 2.6 / c_water)

w.con5 = Constraint(expr=w.x_i['oxygen'] <= m_oxygen * 1.5 / c_oxygen)
w.con6 = Constraint(expr=w.x_i['hydrogen'] <= m_hydrogen * 1 / c_hydrogen)
w.con7 = Constraint(expr=w.x_i['power-solar'] <= m_power_solar / c_power)
# w.con6 = Constraint(expr=w.x <= min(2.6 / m_water*water, 1.5 / m_oxygen*oxygen))


w.obj = Objective(
    expr=sum(w.x_i[i] for i in ['oxygen', 'hydrogen'])
    - sum(w.x_o[i] for i in ['solar', 'wind', 'water'])
)
res_w = solver.solve(w)

print('cost:', w.obj())
print('cost-c', w.x_o['solar']() + w.x_o['wind']() + w.x_o['water']())
# print('cost-p', w.x_i['p']())
print('cost-d', w.x_i['hydrogen']() + w.x_i['oxygen']())

cost: -0.3974141677531508
cost-c 0.3974141677531508
cost-d 0.0


In [54]:
print('hydrogen discharged:', d_hydrogen)
print('oxygen discharged:', d_oxygen)
print('water consumed:', c_water)
print('solar consumed:', c_solar)
print('wind consumed:', c_wind)
print('power produced:', p_power)

hydrogen discharged: 0.38297872340425526
oxygen discharged: 1.7999999999999998
water consumed: 1.129787234042553
solar consumed: 0.09000000000000001
wind consumed: 0.13
power produced: 1.4936170212765956
